# Ejecutor de Modelos HJB y Estocásticos
Este notebook permite inicializar y resolver los modelos **HJB** y **Robust HJB**, así como simular y visualizar sus resultados de manera interactiva.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import logging
import time
import sys
import os

# Añadir directorio raíz para importar los módulos de lib
sys.path.append(os.path.abspath('..'))

from lib.prime_kernel.hjb_solver import (
    GridFrequencyDynamics,
    HJBSolver,
    RobustHJBSolver,
    ISOMarket
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)-7s  %(message)s")
logger = logging.getLogger("hjb_executor")

## 1. Inicialización de los Modelos
Aquí se configura la dinámica del grid y se inicializan ambos solvers: HJB estándar y Robust HJB.

In [ ]:
# Dinámica del mercado seleccionado
dynamics = GridFrequencyDynamics(market=ISOMarket.CENACE)

# Configuración del grid y parámetros de resolución
grid_points = [20, 20]  # Puede aumentarse para mayor precisión (ej. [50, 50])
max_sweeps = 5
epsilon_robust = 0.00346

base_solver = HJBSolver(dynamics, grid_points=grid_points, stochastic=True, max_sweeps=max_sweeps)
robust_solver = RobustHJBSolver(dynamics, epsilon=epsilon_robust, grid_points=grid_points, max_sweeps=max_sweeps)

logger.info("Solvers inicializados correctamente.")

## 2. Resolución de las Ecuaciones de Hamilton-Jacobi-Bellman
Ejecución de los métodos de resolución para encontrar la función de valor óptima.

In [ ]:
logger.info("Resolviendo HJB Base...")
t0 = time.time()
base_solver.solve()
logger.info(f"HJB Base resuelto en {time.time() - t0:.2f} segundos.\n")

logger.info("Resolviendo Robust HJB...")
t1 = time.time()
robust_solver.solve()
logger.info(f"Robust HJB resuelto en {time.time() - t1:.2f} segundos.")

## 3. Simulación de Trayectorias y Evaluación del Control
Una vez calculada la política óptima, procedemos a simular el comportamiento del sistema desde un estado inicial.

In [ ]:
# Estado inicial: [desviación de frecuencia (df), Potencia (P)]
estado_inicial = np.array([0.0, 50.0])

logger.info("Simulando trayectoria con control HJB Base...")
res_base = base_solver.simulate(estado_inicial)

logger.info("Simulando trayectoria con control Robust HJB...")
res_robust = robust_solver.simulate(estado_inicial)

print(f"\nCosto Total (Base): {res_base.total_cost:.4f}")
print(f"Costo Total (Robusto): {res_robust.total_cost:.4f}")

## 4. Visualización de Resultados

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Gráfica Base
traj_base = res_base.state_trajectory
ax[0].plot(traj_base[:, 0], traj_base[:, 1], label='Trayectoria', color='blue')
ax[0].scatter([estado_inicial[0]], [estado_inicial[1]], color='green', zorder=5, label='Estado Inicial')
ax[0].set_title("HJB Base - Trayectoria")
ax[0].set_xlabel("Desviación de Frecuencia (df)")
ax[0].set_ylabel("Potencia (P)")
ax[0].legend()
ax[0].grid(True)

# Gráfica Robusta
traj_robust = res_robust.state_trajectory
ax[1].plot(traj_robust[:, 0], traj_robust[:, 1], label='Trayectoria', color='red')
ax[1].scatter([estado_inicial[0]], [estado_inicial[1]], color='green', zorder=5, label='Estado Inicial')
ax[1].set_title("Robust HJB - Trayectoria")
ax[1].set_xlabel("Desviación de Frecuencia (df)")
ax[1].set_ylabel("Potencia (P)")
ax[1].legend()
ax[1].grid(True)

plt.tight_layout()
plt.show()